# 03 PDF CV Extraction

This notebook tests text extraction from a real CV in PDF format.

Main goals:
- load a PDF CV from the local project folder
- extract text using `pypdf`
- extract text using `pdfplumber`
- compare extraction quality
- clean extracted CV text
- save the final extracted text for later CV quality analysis and structured extraction

## 1. Imports and Settings

In [1]:
import re
import pandas as pd

from pathlib import Path
from pypdf import PdfReader
import pdfplumber

In [2]:
RAW_TEST_CV_DIR = Path("../data/raw/test_cv")
PROCESSED_TEST_CV_DIR = Path("../data/processed/test_cv")

test_cv="Marko Mijanovic CV.pdf"

#test_cv="CV-ks.pdf"

In [3]:
cv_path=RAW_TEST_CV_DIR/test_cv
print(cv_path)

..\data\raw\test_cv\Marko Mijanovic CV.pdf


## 2. Extract Text Using pypdf

`pypdf` is a simple PDF text extraction library.  
It works well for many standard PDFs, but may struggle with complex layouts, tables or multi-column CVs.

In [4]:
def extract_text_with_pypdf(pdf_path):
    
    reader = PdfReader(pdf_path)
    extracted_pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        page_text = page.extract_text()


        if page_text is None:
            page_text = ""

        extracted_pages.append(page_text)

    full_text = "\n\n".join(extracted_pages)

    return full_text

In [5]:
pypdf_text=extract_text_with_pypdf(cv_path)

In [6]:
print(pypdf_text)

+ 3 8 1 6 1 2 9 1 4 2 2 5  |             m a r k o m i j a 8 @ g m a i l . c o m  |         B e l g r a d e ,  1 1 0 6 0 ,  S e r b i a
M A R K O  M I J A N O V I Ć
S O F T W A R E  E N G I N E E R I N G  S T U D E N T
P R O F I L E
U n i v e r s i t y  o f  B e l g r a d e  -  S c h o o l  o f  E l e c t r i c a l  E n g i n e e r i n g  ( E T F )
D e g r e e :  B a c h e l o r  o f  S c i e n c e  ( B . S c . )  i n  S o f t w a r e  E n g i n e e r i n g
L o c a t i o n :  B e l g r a d e ,  S e r b i a
E x p e c t e d  G r a d u a t i o n :  J u n e  2 0 2 7
Y e a r  o f  S t u d y :  T h i r d  Y e a r  S t u d e n t
C u m u l a t i v e  G P A :  9 . 2  /  1 0 . 0
R e l e v a n t  C o u r s e w o r k  H i g h l i g h t s :
D a t a  S t r u c t u r e s  a n d  A l g o r i t h m s  
O b j e c t - O r i e n t e d  P r o g r a m m i n g  1  ( C + + )
O b j e c t - O r i e n t e d  P r o g r a m m i n g  2  ( J a v a )
O p e r a t i n g  S y s t e m s
D a t a b a s e  S y s t e m s  ( 

## 3. Extract Text Using pdfplumber

`pdfplumber` is often better for CVs because it can preserve layout and line structure more accurately than basic PDF extractors.

In [7]:
def extract_text_with_pdfplumber(pdf_path):

    extracted_pages = []


    with pdfplumber.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text()

            if page_text is None:
                page_text = ""

            extracted_pages.append(page_text)

    full_text = "\n\n".join(extracted_pages)

    return full_text

In [8]:
pdfplumber_text=extract_text_with_pdfplumber(cv_path)

In [9]:
print(pdfplumber_text)

MARKO MIJANOVIĆ
+381612914225 | markomija8@gmail.com | Belgrade, 11060, Serbia
SOFTWARE ENGINEERING STUDENT
https://www.linkedin.com/in/marko-mijanovic-16b530315/
PROFILE
Highly motivated Third-Year Software Engineering student (GPA 9.2/10.0) at the School of Electrical
Engineering (ETF), University of Belgrade, specializing in Data Structures and Algorithms (DSA) and
System Programming (C/C++, Java,Python). Seeking to leverage advanced technical skills and a robust
background in low-level systems and application development to contribute high-impact solutions to a
challenging and innovative engineering environment. Strong communication and teamwork skills
developed through university projects.
EDUCATION
University of Belgrade - School of Electrical Engineering (ETF)
Degree: Bachelor of Science (B.Sc.) in Software Engineering
Location: Belgrade, Serbia
Expected Graduation: June 2027
Year of Study: Third Year Student
Cumulative GPA: 9.2 / 10.0
Relevant Coursework Highlights:
Data Struct

## 4. Compare Extraction Results

Both extraction methods are compared using basic text statistics.

The final method should be selected based on:
- text length
- readability
- preservation of sections
- whether skills, education and experience are visible

In [10]:
comparison = pd.DataFrame({
    "method": ["pypdf", "pdfplumber"],
    "character_count": [len(pypdf_text), len(pdfplumber_text)],
    "word_count": [len(pypdf_text.split()), len(pdfplumber_text.split())],
    "line_count": [len(pypdf_text.splitlines()), len(pdfplumber_text.splitlines())]
})



In [11]:
comparison

,method,character_count,word_count,line_count
0,pypdf,4468,2099,53
1,pdfplumber,2399,301,45


Although `pypdf` extracted a higher number of characters and words, the `pdfplumber` output was selected as the final extraction method because it produced a more readable and better structured CV text. Since all relevant CV information was preserved, readability and structure were prioritized over raw text length.

In [12]:
 raw_cv_text = pdfplumber_text

## 5. Clean Extracted CV Text

In [13]:
def clean_extracted_cv_text(text):
 
    if text is None:
        return ""

    text = str(text)

    text = text.replace("\x00", " ")

    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:

        line = re.sub(r"[ \t]+", " ", line)

        line = line.strip()

        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    text = re.sub(r"\n{3,}", "\n\n", text)

    text = text.strip()

    return text

In [14]:
clean_cv_text = clean_extracted_cv_text(raw_cv_text)


In [15]:

print(clean_cv_text)

MARKO MIJANOVIĆ
+381612914225 | markomija8@gmail.com | Belgrade, 11060, Serbia
SOFTWARE ENGINEERING STUDENT
https://www.linkedin.com/in/marko-mijanovic-16b530315/
PROFILE
Highly motivated Third-Year Software Engineering student (GPA 9.2/10.0) at the School of Electrical
Engineering (ETF), University of Belgrade, specializing in Data Structures and Algorithms (DSA) and
System Programming (C/C++, Java,Python). Seeking to leverage advanced technical skills and a robust
background in low-level systems and application development to contribute high-impact solutions to a
challenging and innovative engineering environment. Strong communication and teamwork skills
developed through university projects.
EDUCATION
University of Belgrade - School of Electrical Engineering (ETF)
Degree: Bachelor of Science (B.Sc.) in Software Engineering
Location: Belgrade, Serbia
Expected Graduation: June 2027
Year of Study: Third Year Student
Cumulative GPA: 9.2 / 10.0
Relevant Coursework Highlights:
Data Struct

## 6. Save Extracted CV Text

In [ ]:
output_txt_path = PROCESSED_TEST_CV_DIR / "test_cv_extracted_text.txt"

with open(output_txt_path, "w", encoding="utf-8") as file:
    file.write(clean_cv_text)

print(f"Extracted CV text saved to: {output_txt_path}")